# 在 Kaggle 使用 Dual T4 GPUs 从零训练中文大模型

本笔记本包含以下关键特性：
1. **模型架构**: GPT-2 Small (124M参数)，中文词表大小 21,128，且使用 Weight Tying 技术节省显存。
2. **双卡 DDP 加速**: 使用 PyTorch `DistributedDataParallel` 充分释放两张 T4 显卡算力，支持多进程并行训练。
3. **数据读取**: 使用内存映射文件 `train.bin` 进行流式加载，支持百万至千万级别的中文维基百科（Wikipedia-zh）语料训练。
4. **Kaggle 9小时超时防线**: 自动在运行满 8.3 小时前保存最终 checkpoint 并正常退出，避免因超时被系统清理导致产物丢失。

In [ ]:
# 1. 安装与检查依赖环境
!pip install -q transformers datasets pandas pyarrow tqdm
!nvidia-smi

In [ ]:
%%writefile config.py
import torch

class Config:
    # 模型参数
    vocab_size = 21128  # bert-base-chinese vocab size
    block_size = 512    # 训练时可以先设为512节省显存，正式训练对齐gpt2-small用1024
    n_layer = 12
    n_head = 12
    n_embd = 768
    
    # 训练参数
    batch_size = 8      # 根据显存调整
    learning_rate = 6e-4
    max_iters = 100000
    eval_interval = 500
    save_interval = 2000
    eval_iters = 200
    
    # Kaggle 路径
    tokenizer_path = "bert-base-chinese"
    data_path = "train_data.txt" # 预处理后的文本文件
    device = "cuda" if torch.cuda.is_available() else "cpu"
    checkpoint_path = "gpt2_zh_latest.pt"


In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
from torch.nn import functional as F
import math

class GPTConfig:
    def __init__(self, vocab_size=21128, block_size=1024, n_layer=12, n_head=12, n_embd=768):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # Key, Query, Value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        # Regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # Causal mask to ensure that attention is only applied to the left in the input sequence
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # Calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # Causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # Output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight tying: share weights between embedding and output head
        self.transformer.wte.weight = self.lm_head.weight

        # Init all weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        # Forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = tok_emb + pos_emb
        
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # If we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # Inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=20):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the probabilities to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [ ]:
%%writefile prepare_data.py
import os
import requests
import pandas as pd
import numpy as np
from transformers import BertTokenizer
from tqdm import tqdm
from config import Config

def download_and_build(max_tokens=50_000_000):
    conf = Config()
    tokenizer = BertTokenizer.from_pretrained(conf.tokenizer_path)
    all_tokens = []
    
    # Hugging Face hosted Parquet shards for Wikipedia zh 20231101
    shards = [
        "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/20231101.zh/train-00000-of-00006.parquet",
        "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/20231101.zh/train-00001-of-00006.parquet",
        "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/20231101.zh/train-00002-of-00006.parquet",
        "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/20231101.zh/train-00003-of-00006.parquet",
        "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/20231101.zh/train-00004-of-00006.parquet",
        "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/20231101.zh/train-00005-of-00006.parquet"
    ]

    print(f"Starting to download Chinese Wikipedia shards. Token limit: {max_tokens:,}")
    
    for i, url in enumerate(shards):
        if len(all_tokens) >= max_tokens:
            print(f"Reached token limit of {max_tokens:,}. Stopping download.")
            break
            
        local_parquet = f"wiki_zh_{i}.parquet"
        
        # Download shard
        if not os.path.exists(local_parquet):
            print(f"Downloading shard {i+1}/{len(shards)} from {url}...")
            try:
                r = requests.get(url, stream=True, timeout=30)
                r.raise_for_status()
                with open(local_parquet, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            except Exception as e:
                print(f"Failed to download shard {i}: {e}")
                # Fallback to local files if already downloaded or fail gracefully
                if not os.path.exists(local_parquet):
                    continue
        
        # Load and tokenize
        print(f"Processing shard {i+1}...")
        try:
            df = pd.read_parquet(local_parquet)
            texts = df['text'].tolist()
            
            for text in tqdm(texts, desc=f"Tokenizing shard {i}"):
                if len(text) > 10:
                    ids = tokenizer.encode(text, add_special_tokens=False)
                    all_tokens.extend(ids)
                    if len(all_tokens) >= max_tokens:
                        print(f"Reached token limit of {max_tokens:,} during processing.")
                        break
        except Exception as e:
            print(f"Error processing shard {i}: {e}")
        finally:
            # Clean up disk space
            if os.path.exists(local_parquet):
                os.remove(local_parquet)
                print(f"Removed temporary file {local_parquet}")

    print(f"Preprocessing completed! Total tokens collected: {len(all_tokens):,}")
    
    # Save tokens to train.bin
    if len(all_tokens) > 0:
        np.array(all_tokens, dtype=np.uint16).tofile("train.bin")
        print("--- SUCCESS: train.bin is ready! ---")
    else:
        # Emergency fallback data if completely offline or empty
        print("!!! Warning: No tokens collected. Creating a small dummy dataset to prevent crash !!!")
        fallback_texts = [
            "人工智能是引领未来的战略性技术。",
            "GPT模型通过深度学习掌握了语言的规律。",
            "我们在Kaggle平台上使用双卡GPU进行大语言模型分布式训练。"
        ] * 1000
        for text in fallback_texts:
            all_tokens.extend(tokenizer.encode(text, add_special_tokens=False))
        np.array(all_tokens, dtype=np.uint16).tofile("train.bin")
        print("--- Fallback train.bin created successfully! ---")

if __name__ == "__main__":
    download_and_build()


In [ ]:
%%writefile train_ddp.py
import os
import sys
import time
import math
import torch
import numpy as np
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group
from transformers import BertTokenizer
from model import GPT, GPTConfig
from config import Config

def train():
    # Setup DDP variables
    ddp = int(os.environ.get('RANK', -1)) != -1
    if ddp:
        init_process_group(backend='nccl')
        ddp_rank = int(os.environ['RANK'])
        ddp_local_rank = int(os.environ['LOCAL_RANK'])
        ddp_world_size = int(os.environ['WORLD_SIZE'])
        device = f'cuda:{ddp_local_rank}'
        torch.cuda.set_device(device)
        master_process = ddp_rank == 0
        seed_offset = ddp_rank
    else:
        ddp_rank = 0
        ddp_local_rank = 0
        ddp_world_size = 1
        master_process = True
        seed_offset = 0
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Set random seeds for reproducibility and diversity across ranks
    torch.manual_seed(1337 + seed_offset)
    torch.cuda.manual_seed_all(1337 + seed_offset)
    np.random.seed(1337 + seed_offset)

    conf = Config()
    model_config = GPTConfig(
        vocab_size=conf.vocab_size,
        block_size=conf.block_size,
        n_layer=conf.n_layer,
        n_head=conf.n_head,
        n_embd=conf.n_embd
    )

    if master_process:
        print("Initializing model...")
    model = GPT(model_config).to(device)

    # Wrap model with DDP
    if ddp:
        model = DDP(model, device_ids=[ddp_local_rank])

    # Load data
    if not os.path.exists("train.bin"):
        if master_process:
            print("Error: train.bin not found. Please run prepare_data.py first.")
        if ddp:
            destroy_process_group()
        sys.exit(1)
        
    data = np.memmap('train.bin', dtype=np.uint16, mode='r')
    if master_process:
        print(f"Data has {len(data):,} tokens.")

    def get_batch():
        max_idx = len(data) - conf.block_size - 1
        # Each rank will sample different sequences because their random seeds are offset
        ix = torch.randint(0, max_idx, (conf.batch_size,))
        x = torch.stack([torch.from_numpy((data[i:i+conf.block_size]).astype(np.int64)) for i in ix])
        y = torch.stack([torch.from_numpy((data[i+1:i+1+conf.block_size]).astype(np.int64)) for i in ix])
        return x.to(device), y.to(device)

    # Optimizer & Scaler
    optimizer = torch.optim.AdamW(model.parameters(), lr=conf.learning_rate, weight_decay=0.1)
    scaler = torch.amp.GradScaler('cuda')

    # Load checkpoint if exists
    start_step = 0
    if os.path.exists(conf.checkpoint_path):
        if master_process:
            print(f"Loading checkpoint from {conf.checkpoint_path}...")
        try:
            checkpoint = torch.load(conf.checkpoint_path, map_location=device, weights_only=False)
        except TypeError:
            checkpoint = torch.load(conf.checkpoint_path, map_location=device)
        raw_model = model.module if ddp else model
        raw_model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        start_step = checkpoint['step'] + 1
        if master_process:
            print(f"Resumed training from step {start_step}")

    # Cosine learning rate scheduler with warmup
    max_iters = conf.max_iters
    warmup_iters = 1000
    learning_rate = conf.learning_rate
    min_lr = learning_rate / 10.0

    def get_lr(it):
        if it < warmup_iters:
            return learning_rate * it / warmup_iters
        if it > max_iters:
            return min_lr
        decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
        coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
        return min_lr + coeff * (learning_rate - min_lr)

    tokenizer = BertTokenizer.from_pretrained(conf.tokenizer_path)

    # Time tracking for Kaggle timeout guard (8.3 hours = 29,880 seconds)
    start_time = time.time()
    max_training_seconds = 8 * 3600 + 20 * 60 

    if master_process:
        print("Starting training loop...")
        
    model.train()
    for step in range(start_step, max_iters):
        t0 = time.time()
        
        lr = get_lr(step)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
            
        x, y = get_batch()
        
        with torch.amp.autocast('cuda'):
            logits, loss = model(x, y)
            
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        # Logging
        if step % 50 == 0:
            loss_val = loss.item()
            if ddp:
                # Average loss across all DDP ranks for logging
                torch.distributed.all_reduce(loss, op=torch.distributed.ReduceOp.SUM)
                loss_val = loss.item() / ddp_world_size
                
            if master_process:
                dt = (time.time() - t0) * 1000
                print(f"Step {step} | Loss {loss_val:.4f} | LR {lr:.2e} | Time {dt:.2f}ms")
                
        # Eval & Save
        if step > start_step and step % conf.eval_interval == 0:
            if master_process:
                model.eval()
                raw_model = model.module if ddp else model
                # Check performance via sample generation
                context = torch.tensor([tokenizer.encode("人工智能", add_special_tokens=False)], device=device)
                gen = raw_model.generate(context, max_new_tokens=40)
                print(f"\n--- Step {step} Gen Test: {tokenizer.decode(gen[0].tolist())} ---\n")
                
                # Save checkpoint
                checkpoint = {
                    'model': raw_model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'step': step,
                    'config': model_config
                }
                torch.save(checkpoint, conf.checkpoint_path)
                print(f"Saved checkpoint to {conf.checkpoint_path}")
                model.train()
                
        # Safe exit guard: check elapsed time
        elapsed = time.time() - start_time
        if elapsed > max_training_seconds:
            if master_process:
                print(f"\n[Timeout Guard] Training time elapsed: {elapsed/3600:.2f} hours.")
                print("Saving final checkpoint and exiting to prevent Kaggle timeout cleanup...")
                raw_model = model.module if ddp else model
                checkpoint = {
                    'model': raw_model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'step': step,
                    'config': model_config
                }
                torch.save(checkpoint, conf.checkpoint_path)
                print(f"Checkpoint saved to {conf.checkpoint_path}. Exit successful.")
            if ddp:
                destroy_process_group()
            sys.exit(0)

    # If training completes naturally
    if master_process:
        print("Training reached max iterations!")
        raw_model = model.module if ddp else model
        checkpoint = {
            'model': raw_model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'step': max_iters - 1,
            'config': model_config
        }
        torch.save(checkpoint, conf.checkpoint_path)
        print("Final checkpoint saved.")

    if ddp:
        destroy_process_group()

if __name__ == "__main__":
    train()


## 2. 数据获取与中文分词预处理

运行 `prepare_data.py`。该脚本会从 Hugging Face 下载中文维基百科 Parquet 数据分片，分词转换后输出成 `train.bin` 二进制映射文件。

*注意：Kaggle 环境下在线下载极快。由于是第一次测试运行，可以在下方传入参数调整 token 的限制量，如在 prepare_data.py 修改 `max_tokens` 参数。*

In [ ]:
# 运行数据预处理
!python prepare_data.py

## 3. 双卡分布式训练 (DDP)

我们使用 PyTorch 提供的 `torchrun` 工具在一台机器的多张 GPU 上启动分布式训练。针对 Kaggle 的 2x T4 GPUs，设置 `--nproc_per_node=2`。

In [ ]:
# 启动分布式训练
!torchrun --nproc_per_node=2 train_ddp.py

## 4. 模型生成与推理测试

加载保存好的 Checkpoint 并直接使用 `generate` 接口进行中文内容续写测试。

In [ ]:
import os
import torch
from transformers import BertTokenizer
from model import GPT, GPTConfig
from config import Config

conf = Config()
tokenizer = BertTokenizer.from_pretrained(conf.tokenizer_path)

model_config = GPTConfig(
    vocab_size=conf.vocab_size,
    block_size=conf.block_size,
    n_layer=conf.n_layer,
    n_head=conf.n_head,
    n_embd=conf.n_embd
)
model = GPT(model_config).to(conf.device)

if os.path.exists(conf.checkpoint_path):
    try:
        checkpoint = torch.load(conf.checkpoint_path, map_location=conf.device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(conf.checkpoint_path, map_location=conf.device)
    model.load_state_dict(checkpoint['model'])
    print("Loaded checkpoint successfully!")
    
    model.eval()
    context_text = "人工智能是"
    print(f"输入提示词: {context_text}")
    context = torch.tensor([tokenizer.encode(context_text, add_special_tokens=False)], device=conf.device)
    gen = model.generate(context, max_new_tokens=100, temperature=0.8, top_k=20)
    print(f"模型续写结果:\n{tokenizer.decode(gen[0].tolist())}")
else:
    print(f"未找到 checkpoint 文件: {conf.checkpoint_path}")